# Deploy CPU Inference Endpoint

This notebook does not train the model. It deploys the already-published model repository as a protected Hugging Face Inference Endpoint, enables automatic scale-to-zero, runs one end-to-end smoke test, and prints the server-side integration URL.


In [ ]:
# 1. Prepare the deployment client
import importlib.util, json, subprocess, sys, time
from pathlib import Path

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.35,<2"], check=True)

from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is missing from Colab Secrets")

api = HfApi(token=hf_token)
identity = api.whoami()
namespace = identity.get("name")
if not namespace:
    raise RuntimeError("Could not resolve the Hugging Face namespace")

MODEL_REPO = f"{namespace}/semantic-integrity-v0-1"
ENDPOINT_NAME = "semantic-integrity-v0-1-cpu"
print("Authenticated account:", namespace)
print("Model repository:", MODEL_REPO)


In [ ]:
# 2. Select the cheapest available CPU configuration with at least x4 capacity
hardware = api.list_inference_endpoints_hardware(namespace=namespace)
size_rank = {"x1": 1, "x2": 2, "x4": 4, "x8": 8, "x16": 16, "x32": 32}
candidates = [
    hw for hw in hardware
    if getattr(hw, "accelerator", None) == "cpu"
    and getattr(hw, "status", None) == "available"
    and size_rank.get(getattr(hw, "instance_size", ""), 0) >= 4
]
if not candidates:
    raise RuntimeError("No available CPU x4-or-larger Inference Endpoint quota was found for this account.")

selected = min(candidates, key=lambda hw: float(hw.price_per_hour))
print("Selected hardware:")
print(" provider:", selected.vendor)
print(" region:", selected.region)
print(" instance:", selected.instance_type, selected.instance_size)
print(" hourly USD:", selected.price_per_hour)
print(" estimated 15-minute compute USD:", round(float(selected.price_per_hour) / 4, 4))


In [ ]:
# 3. Create or reuse the protected endpoint with automatic scale-to-zero
repo_info = api.repo_info(repo_id=MODEL_REPO, repo_type="model")
revision = repo_info.sha

existing = {ep.name: ep for ep in api.list_inference_endpoints(namespace=namespace)}
if ENDPOINT_NAME in existing:
    endpoint = existing[ENDPOINT_NAME]
    print("Existing endpoint found:", endpoint.status)
    if str(endpoint.status).lower() == "paused":
        endpoint.resume()
else:
    try:
        endpoint = api.create_inference_endpoint(
        name=ENDPOINT_NAME,
        repository=MODEL_REPO,
        framework="pytorch",
        task="custom",
        accelerator="cpu",
        instance_size=selected.instance_size,
        instance_type=selected.instance_type,
        region=selected.region,
        vendor=selected.vendor,
        namespace=namespace,
        revision=revision,
        min_replica=0,
        max_replica=1,
        scale_to_zero_timeout=15,
        type="authenticated",
            token=hf_token,
        )
    except Exception as exc:
        message = str(exc)
        if "Payment method required" in message or "payment method" in message.lower():
            raise RuntimeError(
                "Hugging Face requires a valid payment method before creating a dedicated Inference Endpoint. "
                "Add a credit card in https://huggingface.co/settings/billing, then rerun only this cell."
            ) from exc
        raise
    print("Endpoint created:", ENDPOINT_NAME)

print("Waiting for endpoint startup...")
endpoint.wait(timeout=1200, refresh_every=10)
print("Endpoint status:", endpoint.status)
print("AMANAH_ML_URL:", endpoint.url)


In [ ]:
# 4. Run a real end-to-end inference smoke test
import requests

reference_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename="reference_store.json",
    repo_type="model",
    token=hf_token,
)
reference_store = json.loads(Path(reference_path).read_text(encoding="utf-8"))
row = reference_store["ayahs"][0]
payload = {
    "inputs": {
        "source_type": "quran",
        "source_ar": row["source_ar"],
        "candidate_en": row["references_en"][0],
        "ayah_id": row["ayah_id"],
    }
}

response = requests.post(
    endpoint.url,
    headers={"Authorization": f"Bearer {hf_token}", "Content-Type": "application/json"},
    json=payload,
    timeout=180,
)
print("HTTP status:", response.status_code)
if response.status_code != 200:
    print(response.text[:4000])
    raise RuntimeError("Live endpoint smoke test failed")

result = response.json()
required_keys = {"decision", "integrity_score", "severity", "confidence", "drifts", "needs_human_review", "model_version", "reference_status"}
missing = sorted(required_keys - set(result))
assert not missing, f"Endpoint response missing keys: {missing}"
assert result["decision"] in {"PASS", "REVIEW", "CRITICAL", "ABSTAIN"}
assert result["reference_status"] == "verified"

print(json.dumps(result, ensure_ascii=False, indent=2))
print("LIVE ENDPOINT SMOKE TEST: PASS")


In [ ]:
# 5. Save integration details without exposing the access token
endpoint_info = {
    "endpoint_name": ENDPOINT_NAME,
    "endpoint_url": endpoint.url,
    "repository": MODEL_REPO,
    "revision": revision,
    "provider": selected.vendor,
    "region": selected.region,
    "accelerator": "cpu",
    "instance_type": selected.instance_type,
    "instance_size": selected.instance_size,
    "price_per_hour_usd": float(selected.price_per_hour),
    "scale_to_zero_minutes": 15,
    "max_replicas": 1,
    "smoke_test": "pass",
}
Path("/content/AMANAH_ENDPOINT_INFO.json").write_text(json.dumps(endpoint_info, indent=2), encoding="utf-8")

print("\nSERVER-SIDE SETTINGS FOR THE TEAM")
print("AMANAH_ML_URL=" + endpoint.url)
print("AMANAH_ML_TOKEN=<use a server-side Hugging Face token; never put it in frontend code>")
print("AMANAH transport=hf")
print("\nAutomatic scale-to-zero: 15 minutes")
print("The endpoint automatically wakes on the next request, with a cold-start delay.")
